# HydroSeason: Fitzroy/Kimberley AOI — DEA STAC WOfS Run

End-to-end run against a **real AOI** (`data/fitzroy_kimberley_aoi.geojson`, Fitzroy River near Derby, Kimberley WA) using the **Digital Earth Australia (DEA) STAC** catalog for WOfS (Water Observations from Space).

Pipeline:
1. Load AOI polygon.
2. Query + load monthly WOfS composites from DEA STAC (`load_wofs_from_stac`), clipped to the AOI.
3. Summarise monthly water extent (`monthly_water_extent`).
4. Detect hydrological years (`detect_hydrological_years`).
5. Label months Wet/Dry (`label_hydrological_months`).
6. Plot + generate interactive HTML report (`generate_html_report`).

Requires the `stac` extra: `pip install hydroseason[stac]` (pulls in `pystac-client`, `odc-stac`, `rioxarray`, `geopandas`, `dask`).

In [ ]:
from __future__ import annotations

import os

# A system-wide PROJ_LIB pointing at an incompatible proj.db (e.g. from a
# PostGIS install) breaks pyproj/rasterio CRS lookups; force pyproj's own
# bundled database instead.
os.environ.pop('PROJ_LIB', None)
os.environ.pop('PROJ_DATA', None)

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from hydroseason import (
    HydroYearConfig,
    detect_hydrological_years,
    label_hydrological_months,
    load_aoi,
    load_wofs_from_stac,
    monthly_water_extent,
    generate_html_report,
)

print('Imports successful!')

## 1. Configuration

Single input block: AOI path, STAC endpoint/collection, and the date range for the analysis.

In [ ]:
AOI_PATH = Path('../data/fitzroy_kimberley_aoi.geojson')

STAC_URL = 'https://explorer.dea.ga.gov.au/stac'
COLLECTION = 'ga_ls_wo_3'  # Landsat WOfS, per-scene water classification

START_DATE = '2015-01-01'
END_DATE = '2025-12-31'

OUTPUT_CRS = 3577  # GDA94 / Australian Albers, matches WOFS pipeline contract

REPORT_PATH = Path('hydroseason_fitzroy_kimberley_report.html')

## 2. Load AOI

Reads and validates the AOI polygon (non-empty, valid geometry).

In [ ]:
aoi_gdf = load_aoi(AOI_PATH)
print(f'AOI features: {len(aoi_gdf)}')
print(f'AOI CRS: {aoi_gdf.crs}')
print(f'AOI bounds (lon/lat): {aoi_gdf.total_bounds}')
aoi_gdf.plot(edgecolor='#0284c7', facecolor='#e0f2fe', figsize=(6, 6))
plt.title('Fitzroy / Kimberley AOI')
plt.show()

## 3. Pull WOfS from DEA STAC

Queries the STAC catalog for the AOI bbox and date range, groups observations by month, composites them (majority vote), and clips every month to the AOI polygon. This is a lazy, dask-backed cube — nothing is computed yet.

In [ ]:
water_mask = load_wofs_from_stac(
    STAC_URL,
    COLLECTION,
    AOI_PATH,
    START_DATE,
    END_DATE,
    crs=OUTPUT_CRS,
)

print(f'Loaded cube: {dict(water_mask.sizes)}')
print(f'Months: {water_mask.sizes["time"]}')
print(f'Inserted (gap-filled) months: {water_mask.attrs.get("n_inserted_timesteps", 0)}')

## 4. Summarise Monthly Water Extent

Computes the dask graph once and returns `extent_pct` / `invalid_pct` per month.

In [ ]:
extent = monthly_water_extent(water_mask)
print(f'Extent series shape: {extent.shape}')
print(f'Max invalid_pct: {extent["invalid_pct"].max():.1f}%')
print(f'Months with invalid_pct > 50%: {(extent["invalid_pct"] > 50).sum()}')
extent.head(12)

## 5. Detect Hydrological Years

Same detection API as the mock-data walkthrough, applied to the real remote-sensing series. Adjust `max_invalid_pct` / `missing_month_policy` after inspecting cloud/gap coverage above.

In [ ]:
config = HydroYearConfig(
    wet_start_month=11,
    wet_end_month=4,
    dry_start_month=7,
    dry_end_month=12,
)

# 2015-01 is the only month above 50% invalid coverage in this AOI (start of
# the Landsat WOfS record has partial scene overlap here); 95% comfortably
# clears it without silently accepting a systematically bad month elsewhere.
hydro_years = detect_hydrological_years(
    extent,
    config=config,
    missing_month_policy='ignore',
    max_invalid_pct=95.0,
)

print(f'Detected {len(hydro_years)} hydrological years:')
hydro_years

## 6. Label Months + Visualise

In [ ]:
labels = label_hydrological_months(extent.index, hydro_years)

plt.figure(figsize=(15, 6))
for ts, row in labels.iterrows():
    color = '#e0f2fe' if row['season'] == 'Wet' else '#fef3c7'
    plt.axvspan(ts - pd.Timedelta(days=15), ts + pd.Timedelta(days=15), color=color, alpha=0.5, linewidth=0)

plt.plot(extent.index, extent['extent_pct'], color='#0284c7', marker='o', markersize=4, label='Monthly Extent (%)')
for _, row in hydro_years.iterrows():
    plt.plot(pd.Timestamp(row['peak_month']), row['peak_extent_pct'], marker='o', color='#10b981', markersize=8, zorder=5)
    plt.plot(pd.Timestamp(row['end_dry_month']), row['end_extent_pct'], marker='o', color='#ef4444', markersize=8, zorder=5)

plt.title('Fitzroy / Kimberley AOI — Hydrological Year Detections', fontsize=14, fontweight='bold')
plt.ylabel('Water Extent Percentage (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(['Monthly Extent', 'Wet Season', 'Dry Season', 'Wet Peak', 'Dry End'], loc='upper right')
plt.tight_layout()
plt.show()

## 7. Generate HTML Report

In [ ]:
generate_html_report(
    extent=extent,
    hydro_years=hydro_years,
    output_path=REPORT_PATH,
    title='HydroSeason — Fitzroy / Kimberley AOI (DEA STAC WOfS)',
)

print(f'HTML report written to: {REPORT_PATH.resolve()}')